In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class NMethylation(MorphingOperator):
    def __init__(self):
        super(NMethylation, self).__init__()
        self._name = "N-methylation (Primary Amines)"
        self._target_atoms = []
        
        # Ενισχυμένο SMARTS Pattern για Πρωτοταγείς Αμίνες:
        # Ζητάμε άζωτο με 2 υδρογόνα [N;H2]
        # Αποκλείουμε: Αμίδια (!$(N-C=O)), Υδραζίνες (!$(NN)), 
        # Σουλφοναμίδια (!$(N-S(=O)=O)), Ουρίες/Καρβαμιδικά (!$(N-C(=O)))
        self.PATTERN = Chem.MolFromSmarts("[N;H2;!$(N-C=O);!$(N-S(=O)=O);!$(NN)]")

    def setOriginal(self, mol):
        super(NMethylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_n_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            # Προσθήκη του Άνθρακα του Μεθυλίου (-CH3)
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(target_n_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            # Reset ιδιοτήτων στα άτομα που τροποποιήθηκαν
            for idx in [target_n_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name
    
nmethylation_op = NMethylation()

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target



start_mol = MolpherMol("CN") # Μεθυλαμίνη (Πρωτοταγής αμίνη)
target_mol = MolpherMol("CNC") # Διμεθυλαμίνη (Στόχος)
tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = (nmethylation_op,)

closest_info = FindClosest()

print("--- STARTING N-METHYLATION TREE SEARCH ---")
while not tree.path_found:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
        
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)
        
    if tree.path_found or tree.generation_count >= 5:
        break

print("\nSearch finished!")
if tree.path_found:
    print("SUCCESS: Target reached! The tree completed the N-Methylation.")


print("\n=== RUNNING FALSE POSITIVE TRAP TESTS ===")
traps = {
        "Secondary Amine Trap": "CNC",
        "Amide Trap (CC(=O)NC)": "CC(=O)NC",
        "Sulfonamide Trap (CCS(=O)(=O)N)": "CCS(=O)(=O)N",
        "Urea Trap (CNC(=O)NC)": "CNC(=O)NC"
}
    
for name, smiles in traps.items():
    mol = MolpherMol(smiles)
    nmethylation_op.setOriginal(mol)
    product = nmethylation_op.morph()
        
    safe = (product.getSMILES() == mol.getSMILES())
    print(f"{name}:")
    print(f"  SOURCE: {mol.getSMILES()}")
    print(f"  STATUS: {'SAFE (Passed)' if safe else 'VULNERABLE (Failed)'}")
    print("-" * 50)

--- STARTING N-METHYLATION TREE SEARCH ---
Generation #1
Molecules in tree: 2
Closest to target: CNC (Distance: 0.0000)
----------------------------------------

Search finished!
SUCCESS: Target reached! The tree completed the N-Methylation.

=== RUNNING FALSE POSITIVE TRAP TESTS ===
Secondary Amine Trap:
  SOURCE: CNC
  STATUS: SAFE (Passed)
--------------------------------------------------
Amide Trap (CC(=O)NC):
  SOURCE: CNC(C)=O
  STATUS: SAFE (Passed)
--------------------------------------------------
Sulfonamide Trap (CCS(=O)(=O)N):
  SOURCE: CCS(N)(=O)=O
  STATUS: SAFE (Passed)
--------------------------------------------------
Urea Trap (CNC(=O)NC):
  SOURCE: CNC(=O)NC
  STATUS: SAFE (Passed)
--------------------------------------------------
